# 1. Import dependencies

In [1]:
!python -m pip install --quiet -r ../requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split

from churn_prediction.paths import PROCESSED_DATA_PATH

df = pd.read_csv(PROCESSED_DATA_PATH)

X = df.drop(
    columns=["customerID", "Churn"]
)

y = df["Churn"].map({
    "No": 0,
    "Yes": 1,
})

## Kiểm tra

In [3]:
print(y.value_counts())
print(y.value_counts(normalize=True))

Churn
0    5174
1    1869
Name: count, dtype: int64
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


# 2. Split dev và test

## Import và split

In [4]:
from sklearn.model_selection import train_test_split
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=69,
    stratify=y,
)

## Kiểm tra

In [5]:
print(X_dev.shape)
print(X_test.shape)

print(y_dev.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(5634, 19)
(1409, 19)
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


# 3. Tạo `StratifiedKFold`

In [6]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=99,
)

## Nhìn các fold

In [7]:
for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_dev, y_dev),
    start=1,
):
    y_train_fold = y_dev.iloc[train_idx]
    y_val_fold = y_dev.iloc[val_idx]
    
    print(f"Fold {fold}")
    
    print(
        "Train size",
        len(train_idx),
    )
    
    print(
        "Validation size",
        len(val_idx)
    )
    
    print("Train distribution:")
    print(
        y_train_fold.value_counts(
            normalize=True
        )
    )
    
    print("Validation distribution:")
    print(
        y_val_fold.value_counts(
            normalize=True
        )
    )
        
    print()

Fold 1
Train size 4507
Validation size 1127
Train distribution:
Churn
0    0.734635
1    0.265365
Name: proportion, dtype: float64
Validation distribution:
Churn
0    0.734694
1    0.265306
Name: proportion, dtype: float64

Fold 2
Train size 4507
Validation size 1127
Train distribution:
Churn
0    0.734635
1    0.265365
Name: proportion, dtype: float64
Validation distribution:
Churn
0    0.734694
1    0.265306
Name: proportion, dtype: float64

Fold 3
Train size 4507
Validation size 1127
Train distribution:
Churn
0    0.734635
1    0.265365
Name: proportion, dtype: float64
Validation distribution:
Churn
0    0.734694
1    0.265306
Name: proportion, dtype: float64

Fold 4
Train size 4507
Validation size 1127
Train distribution:
Churn
0    0.734635
1    0.265365
Name: proportion, dtype: float64
Validation distribution:
Churn
0    0.734694
1    0.265306
Name: proportion, dtype: float64

Fold 5
Train size 4508
Validation size 1126
Train distribution:
Churn
0    0.734694
1    0.265306
Name: 

# 4. Preprocessing

## Import thư viện

In [8]:
from sklearn.preprocessing import(
    OrdinalEncoder,
    OneHotEncoder,
    StandardScaler,
)

from sklearn.compose import ColumnTransformer

## Xác định loại của features

In [9]:
numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]
yes_no_features = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]
binary_features = [
    "SeniorCitizen",
    *yes_no_features
]
multiclass_features = [
    "gender",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

## Kiểm tra

In [10]:
all_features = (
    numerical_features
    + binary_features
    + multiclass_features
)
assert len(all_features) == len(set(all_features))
assert set(all_features) == set(X.columns)

## Binary Transformer

In [11]:
binary_transformer = OrdinalEncoder(
    categories=[
        ["No", "Yes"]
        for _ in yes_no_features
    ]
)

## Preprocessor

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features,
        ),
        (
            "binary",
            binary_transformer,
            yes_no_features,
        ),
        (
            "senior",
            "passthrough",
            ["SeniorCitizen"],
        ),
        (
            "multi",
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False,
            ),
            multiclass_features,
        ),
    ],
    remainder="drop"
)

# 5. Train model

## Logistic Regression

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
            ),
        ),
    ]
)

## Cross validation

In [15]:
from sklearn.model_selection import cross_validate

scores = cross_validate(
    logistic_pipeline,
    X_dev,
    y_dev,
    cv = cv,
    scoring=[
        "roc_auc",
        "average_precision",
        "f1",
    ],
    n_jobs=-1,
)

In [16]:
scores

{'fit_time': array([0.13263202, 0.13220882, 0.13625813, 0.13201809, 0.13145876]),
 'score_time': array([0.02508688, 0.02403927, 0.02162194, 0.03223801, 0.02473712]),
 'test_roc_auc': array([0.85841695, 0.82991817, 0.83667378, 0.85792416, 0.8387693 ]),
 'test_average_precision': array([0.67131203, 0.64767251, 0.62624055, 0.67395529, 0.65116297]),
 'test_f1': array([0.63157895, 0.58450704, 0.57818182, 0.59541985, 0.59312839])}

In [17]:
print(
    "ROC-AUC:",
    scores["test_roc_auc"],
)

print(
    "Average Precision:",
    scores["test_average_precision"],
)

print(
    "F1:",
    scores["test_f1"],
)

ROC-AUC: [0.85841695 0.82991817 0.83667378 0.85792416 0.8387693 ]
Average Precision: [0.67131203 0.64767251 0.62624055 0.67395529 0.65116297]
F1: [0.63157895 0.58450704 0.57818182 0.59541985 0.59312839]


## Tính mean và standard deviation

In [18]:
import numpy as np


print(
    "ROC-AUC:",
    scores["test_roc_auc"].mean(),
    "±",
    scores["test_roc_auc"].std(),
)

print(
    "Average Precision:",
    scores["test_average_precision"].mean(),
    "±",
    scores["test_average_precision"].std(),
)

print(
    "F1:",
    scores["test_f1"].mean(),
    "±",
    scores["test_f1"].std(),
)

ROC-AUC: 0.8443404682430542 ± 0.01166604337145608
Average Precision: 0.654068669979219 ± 0.017415908934942582
F1: 0.5965632091457499 ± 0.018560354909813198


## So sánh với nhiều models

### Import

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)

### Define

In [20]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=36,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=36,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        random_state=36,
    ),
}

### Pipeline

In [21]:
from sklearn.base import clone

pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "model",
                model,
            )
        ]
    )

### Chạy CV

In [22]:
cv_results = []
scoring = [
    "roc_auc",
    "average_precision",
    "f1",
]

for name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_dev,
        y_dev,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )
    
    cv_results.append({
        "model": name,
        
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std(),
        
        "average_precision_mean": scores["test_average_precision"].mean(),
        "average_precision_std": scores["test_average_precision"].std(),

        "f1_mean": scores["test_f1"].mean(),
        "f1_std": scores["test_f1"].std(),
    })

In [23]:
cv_results_df = pd.DataFrame(
    cv_results
)

cv_results_df.sort_values(
    "roc_auc_mean",
    ascending=False,
)

,model,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,f1_mean,f1_std
0,LogisticRegression,0.844340,0.011666,0.654069,0.017416,0.596563,0.018560
3,HistGradientBoosting,0.828936,0.010480,0.644356,0.023771,0.574317,0.015627
2,Random Forest,0.817211,0.011668,0.610040,0.027387,0.540924,0.026630
1,Decision Tree,0.654975,0.014730,0.376870,0.014739,0.494141,0.021477


Trong 5-fold CV hiện tại, Logistic Regression có performance trung bình tốt nhất và khá ổn định; HistGradientBoosting là đối thủ gần nhất.

# 6. OOF

## Cross_val_predict

Dùng cross_val_predict với Logistic vì đang hiện là candidate mạnh nhất

In [25]:
from sklearn.model_selection import cross_val_predict
oof_prob = cross_val_predict(
    logistic_pipeline,
    X_dev,
    y_dev,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

In [26]:
print(oof_prob.shape)
print(X_dev.shape)

(5634,)
(5634, 19)


## Ranking bằng OOF

In [27]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)
oof_roc_auc = roc_auc_score(
    y_dev,
    oof_prob,
)

oof_average_precision = average_precision_score(
    y_dev,
    oof_prob,
)

print(f"ROC-AUC: {oof_roc_auc:.4f}")
print(f"Average Precision: {oof_average_precision:.4f}")

ROC-AUC: 0.8439
Average Precision: 0.6510


In [29]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)

oof_pred_05 = (oof_prob >= 0.5).astype(int)

oof_pred_05[:10]

array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0])

In [30]:
print(
    "Precision:",
    precision_score(
        y_dev,
        oof_pred_05,
        zero_division=0,
    ),
)

print(
    "Recall:",
    recall_score(
        y_dev,
        oof_pred_05,
        zero_division=0,
    ),
)

print(
    "F1:",
    f1_score(
        y_dev,
        oof_pred_05,
        zero_division=0,
    ),
)

Precision: 0.6599025974025974
Recall: 0.5438127090301004
F1: 0.5962596259625963


## Chọn Threshold bằng OOF

In [31]:
def find_best_f1_threshold(
    y_true, 
    y_prob,
):
    rows = []
    
    for threshold in np.arange(
        0.1,
        0.91,
        0.01
    ):
        y_pred = (
            y_prob >= threshold
        ).astype(int)
        
        rows.append({
            "threshold": threshold,
            "precision": precision_score(
                y_true,
                y_pred,
                zero_division=0,
            ),

            "recall": recall_score(
                y_true,
                y_pred,
                zero_division=0,
            ),

            "f1": f1_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
        })
    threshold_df = pd.DataFrame(
        rows
    )
    
    best_row = threshold_df.loc[
        threshold_df["f1"].idxmax()
    ]
    return threshold_df, best_row

In [32]:
threshold_df, best_oof_threshold = find_best_f1_threshold(
    y_dev,
    oof_prob
)

best_oof_threshold

threshold    0.280000
precision    0.529018
recall       0.792642
f1           0.634538
Name: 18, dtype: float64

### Top 10

In [33]:
threshold_df.sort_values(
    "f1",
    ascending=False,
).head(10)

,threshold,precision,recall,f1
18,0.28,0.529018,0.792642,0.634538
17,0.27,0.522004,0.801338,0.632190
19,0.29,0.533119,0.775251,0.631780
20,0.30,0.537020,0.766555,0.631579
16,0.26,0.514843,0.812040,0.630158
21,0.31,0.541386,0.752508,0.629723
22,0.32,0.544466,0.737124,0.626314
15,0.25,0.505780,0.819398,0.625479
27,0.37,0.574635,0.684950,0.624962
28,0.38,0.581128,0.675585,0.624807


## Chọn Threshold

In [35]:
SELECTED_THRESHOLD = 0.28

# Milestone 7 — Cross-validation summary

5-fold Stratified Cross-Validation xác nhận Logistic Regression
có mean ROC-AUC, Average Precision và F1@0.5 tốt nhất trong
các candidate models đã thử.

Logistic CV:
- ROC-AUC: 0.8443 ± 0.0117
- Average Precision: 0.6541 ± 0.0174
- F1@0.5: 0.5966 ± 0.0186

Out-of-fold evaluation:
- ROC-AUC: 0.8439
- Average Precision: 0.6510

Best OOF threshold by F1:
- Threshold: 0.28
- Precision: 0.5290
- Recall: 0.7926
- F1: 0.6345

Therefore:
- Final model candidate: Logistic Regression
- Final decision threshold: 0.28

The threshold is selected from OOF predictions across the
development set, rather than from a single validation split.